In [1]:
import sqlite3
import json
import re

dict_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
entry_db_path = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_entries.db"

# 表名
# - dict_table: 《辞典》条目表
# - entry_table: 官制条目表
dict_table = "chapter8t10"
entry_table = "entries0124"


In [2]:
# 从《辞典》数据库中获取全部条目信息，并据此创建条目表
conn_dict = sqlite3.connect(dict_db_path)
cursor_dict = conn_dict.cursor()
cursor_dict.execute(f"SELECT title, catalog, page FROM {dict_table}")

entry_list = []
entry_indexes = set()
for row in cursor_dict.fetchall():
  entry_list.append({
    "title": row[0],
    "catalog": row[1],
    "page": row[2]
  })
  entry_indexes.add(f"{row[0]}-{row[2]}")
print(len(entry_list), len(entry_indexes))


cursor_dict.close()
conn_dict.close()

# 连接到条目数据库
conn_entry = sqlite3.connect(entry_db_path)
cursor_entry = conn_entry.cursor()

cursor_entry.execute(f"""
DROP TABLE IF EXISTS {entry_table}
""")

cursor_entry.execute(f"""
CREATE TABLE IF NOT EXISTS {entry_table} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    catalog TEXT NOT NULL,
    page TEXT NOT NULL,
    type TEXT,
    subtype TEXT,
    bgn_time TEXT,
    bgn_event TEXT,
    end_time TEXT,
    end_event TEXT,
    superiors TEXT,
    subordinates TEXT,
    staffing TEXT,
    department TEXT,
    grade TEXT
)
""")

# 从 entry_list 插入初始数据
cursor_entry.executemany(f"""
INSERT INTO {entry_table} (title, catalog, page, bgn_time, bgn_event, end_time, end_event, superiors, subordinates, staffing)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", [
    (
        entry["title"],
        entry["catalog"],
        entry["page"],
        "-INF",      # bgn_time 初始值
        "始置",      # bgn_event 初始值
        "INF",       # end_time 初始值
        "罢置",      # end_event 初始值
        "[]",        # superiors 初始值（空列表）
        "[]",        # subordinates 初始值（空列表）
        "[]"         # staffing 初始值（空列表）
    )
    for entry in entry_list
])

conn_entry.commit()

# 验证插入结果
cursor_entry.execute(f"SELECT COUNT(*) FROM {entry_table}")
count = cursor_entry.fetchone()[0]
print(f"成功插入 {count} 条记录到 {entry_table} 表")

# 查看前几条记录
cursor_entry.execute(f'SELECT id, title, catalog, page, bgn_time, bgn_event, end_time, end_event FROM {entry_table} LIMIT 5')
for row in cursor_entry.fetchall():
    print(f"Id: {row[0]}, Title: {row[1]}, Page: {row[3]}, bgn_time: {row[4]}, bgn_event: {row[5]}, end_time: {row[6]}, end_event: {row[7]}")

cursor_entry.close()
conn_entry.close()


833 833
成功插入 833 条记录到 entries0124 表
Id: 1, Title: 河北兵马大元帅府, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 2, Title: 河北兵马大元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 3, Title: 河北兵马元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 4, Title: 河北兵马副元帅, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置
Id: 5, Title: 河北兵马大元帅府参议官, Page: 482, bgn_time: -INF, bgn_event: 始置, end_time: INF, end_event: 罢置


In [3]:
"""
构建信息获取与写入工具
传入参数为 (conn, **kwargs) 其中 conn 为对应数据库的连接对象；
返回值为 JSON 对象：
- 读操作：返回对象或对象列表
- 写操作：返回更新后的对象（或对象列表）
- 错误：返回 {"error": "..."}
"""

# 字段白名单（防 SQL 注入）
_entry_attr_whitelist = {
  "type",
  "subtype",
  "bgn_time",
  "bgn_event",
  "end_time",
  "end_event",
  "superiors",
  "subordinates",
  "staffing",
  "department",
  "grade",
}
_entry_list_attr_whitelist = {"superiors", "subordinates", "staffing"}


def _loads_list(v):
  if v is None or v == "":
    return []
  try:
    x = json.loads(v)
  except Exception:
    return []
  return x if isinstance(x, list) else []


def _dumps_list(v):
  if v is None:
    return "[]"
  if isinstance(v, list):
    return json.dumps(v, ensure_ascii=False)
  # 允许直接传入 JSON 字符串
  if isinstance(v, str):
    try:
      x = json.loads(v)
      if isinstance(x, list):
        return json.dumps(x, ensure_ascii=False)
    except Exception:
      pass
  raise ValueError("列表字段要求 list 或可解析为 list 的 JSON 字符串")


def _row_to_entry_obj(row):
  return {
    "id": row[0],
    "title": row[1],
    "catalog": row[2],
    "page": row[3],
    "type": row[4],
    "subtype": row[5],
    "bgn_time": row[6],
    "bgn_event": row[7],
    "end_time": row[8],
    "end_event": row[9],
    "superiors": _loads_list(row[10]),
    "subordinates": _loads_list(row[11]),
    "staffing": _loads_list(row[12]),
    "department": row[13],
    "grade": row[14],
  }


def _fetch_entry_by_id(cursor, entry_id):
  cursor.execute(
    f"""
    SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
           superiors, subordinates, staffing, department, grade
    FROM {entry_table}
    WHERE id = ?
    """,
    (entry_id,),
  )
  row = cursor.fetchone()
  if row is None:
    return None
  return _row_to_entry_obj(row)


def _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time):
  cursor.execute(
    f"""
    SELECT id
    FROM {entry_table}
    WHERE title = ? AND page = ? AND bgn_time = ? AND end_time = ?
    """,
    (title, str(page), bgn_time, end_time),
  )
  row = cursor.fetchone()
  return None if row is None else row[0]


# 1. 查询《辞典》数据
def search_dictionary(conn, title, page):
  cursor = conn.cursor()
  try:
    cursor.execute(
      f"""
      SELECT id, title, catalog, page, text, fields
      FROM {dict_table}
      WHERE title = ? AND page = ?
      """,
      (title, str(page)),
    )
    row = cursor.fetchone()
    if row is None:
      return {"error": f"未找到条目: {title} (页码: {page})"}

    return {
      "id": row[0],
      "title": row[1],
      "catalog": row[2],
      "page": row[3],
      "text": row[4],
      "fields": json.loads(row[5]) if row[5] else {},
    }
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 2. 查询已有官制条目数据
def check_existing_entry(conn, title, page):
  cursor = conn.cursor()
  try:
    cursor.execute(
      f"""
      SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
             superiors, subordinates, staffing, department, grade
      FROM {entry_table}
      WHERE title = ? AND page = ?
      ORDER BY bgn_time, end_time, id
      """,
      (title, str(page)),
    )

    rows = cursor.fetchall()
    # 按文档约定：查不到返回空列表（不算错误）
    return [_row_to_entry_obj(r) for r in rows]
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 3. 填入官制条目的指定属性（创建）
def insert(conn, title, page, bgn_time, end_time, attr_key, attr_value):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_attr_whitelist:
      return {"error": f"无效的属性名: {attr_key}"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_value = cursor.fetchone()[0]
    if current_value is not None and current_value != "" and current_value != "[]":
      return {"error": f"属性 {attr_key} 已有值: {current_value}，请使用 update"}

    if attr_key in _entry_list_attr_whitelist:
      attr_value = _dumps_list(attr_value)

    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (attr_value, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "写入成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 4. 更新官制条目的指定属性（覆盖）
def update(conn, title, page, bgn_time, end_time, attr_key, attr_value):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_attr_whitelist:
      return {"error": f"无效的属性名: {attr_key}"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    if attr_key in _entry_list_attr_whitelist:
      attr_value = _dumps_list(attr_value)

    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (attr_value, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 5. 插入官制条目的指定列表属性（添加）
def append_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index=None):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index is None:
      current_list.append(attr_value)
    else:
      if not isinstance(index, int):
        return {"error": "index 必须为 int 或 None"}
      if index < 0 or index > len(current_list):
        return {"error": f"index 越界: {index} (len={len(current_list)})"}
      current_list.insert(index, attr_value)

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 6. 修改官制条目的指定列表属性（修改）
def update_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}
    if not isinstance(index, int):
      return {"error": "index 必须为 int"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index < 0 or index >= len(current_list):
      return {"error": f"索引 {index} 超出范围 (列表长度: {len(current_list)})"}

    current_list[index] = attr_value

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 7. 删除官制条目的指定列表属性（删除）
def remove_list(conn, title, page, bgn_time, end_time, attr_key, attr_value, index):
  cursor = conn.cursor()
  try:
    if attr_key not in _entry_list_attr_whitelist:
      return {"error": f"属性 {attr_key} 不是列表类型"}
    if not isinstance(index, int):
      return {"error": "index 必须为 int"}

    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"SELECT {attr_key} FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )
    current_list = _loads_list(cursor.fetchone()[0])

    if index < 0 or index >= len(current_list):
      return {"error": f"索引 {index} 超出范围 (列表长度: {len(current_list)})"}

    if current_list[index] != attr_value:
      return {
        "error": "索引位置的值不匹配，拒绝删除",
        "current": current_list[index],
        "expected": attr_value,
      }

    current_list.pop(index)

    new_list_str = json.dumps(current_list, ensure_ascii=False)
    cursor.execute(
      f"UPDATE {entry_table} SET {attr_key} = ? WHERE id = ?",
      (new_list_str, entry_id),
    )
    conn.commit()

    obj = _fetch_entry_by_id(cursor, entry_id)
    return obj if obj is not None else {"error": "更新成功但读取更新后对象失败"}
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}


# 8. 通过添加时间点的方式，拆分官制条目
def new_time_point(conn, title, page, bgn_time, end_time, time_point, event):
  cursor = conn.cursor()
  try:
    entry_id = _fetch_entry_by_meta(cursor, title, page, bgn_time, end_time)
    if entry_id is None:
      return {"error": f"未找到条目 (title={title}, page={page}, bgn_time={bgn_time}, end_time={end_time})"}

    cursor.execute(
      f"""
      SELECT id, title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
             superiors, subordinates, staffing, department, grade
      FROM {entry_table}
      WHERE id = ?
      """,
      (entry_id,),
    )
    row = cursor.fetchone()
    if row is None:
      return {"error": "读取待拆分条目失败"}

    old = _row_to_entry_obj(row)

    # 插入两条新记录（复制旧记录其他字段）
    cursor.execute(
      f"""
      INSERT INTO {entry_table}
      (title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
       superiors, subordinates, staffing, department, grade)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      """,
      (
        old["title"],
        old["catalog"],
        str(old["page"]),
        old["type"],
        old["subtype"],
        old["bgn_time"],
        old["bgn_event"],
        time_point,
        event,
        json.dumps(old["superiors"], ensure_ascii=False),
        json.dumps(old["subordinates"], ensure_ascii=False),
        json.dumps(old["staffing"], ensure_ascii=False),
        old["department"],
        old["grade"],
      ),
    )
    id1 = cursor.lastrowid

    cursor.execute(
      f"""
      INSERT INTO {entry_table}
      (title, catalog, page, type, subtype, bgn_time, bgn_event, end_time, end_event,
       superiors, subordinates, staffing, department, grade)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
      """,
      (
        old["title"],
        old["catalog"],
        str(old["page"]),
        old["type"],
        old["subtype"],
        time_point,
        event,
        old["end_time"],
        old["end_event"],
        json.dumps(old["superiors"], ensure_ascii=False),
        json.dumps(old["subordinates"], ensure_ascii=False),
        json.dumps(old["staffing"], ensure_ascii=False),
        old["department"],
        old["grade"],
      ),
    )
    id2 = cursor.lastrowid

    # 删除旧记录
    cursor.execute(
      f"DELETE FROM {entry_table} WHERE id = ?",
      (entry_id,),
    )

    conn.commit()

    obj1 = _fetch_entry_by_id(cursor, id1)
    obj2 = _fetch_entry_by_id(cursor, id2)
    if obj1 is None or obj2 is None:
      return {"error": "拆分成功但读取新对象失败"}
    return [obj1, obj2]
  except Exception as e:
    conn.rollback()
    return {"error": str(e)}



In [4]:
"""
Tools 类：封装所有工具函数，管理数据库连接，并提供引用完整性检查
"""

import sqlite3
import re


class Tools:
  def __init__(self, dict_db_path, entry_db_path, indexes):
    """
    初始化 Tools 实例
    
    Args:
      dict_db_path: 《辞典》数据库路径
      entry_db_path: 官制条目数据库路径
      indexes: 索引集合，格式为 set of "名称-页码" 字符串
    """
    self.dict_db_path = dict_db_path
    self.entry_db_path = entry_db_path
    self.indexes = indexes
    
    # 在初始化时建立数据库连接
    self._dict_conn = sqlite3.connect(self.dict_db_path)
    self._entry_conn = sqlite3.connect(self.entry_db_path)
  
  def close(self):
    """关闭所有数据库连接"""
    if self._dict_conn is not None:
      self._dict_conn.close()
      self._dict_conn = None
    if self._entry_conn is not None:
      self._entry_conn.close()
      self._entry_conn = None
  
  def __enter__(self):
    return self
  
  def __exit__(self, exc_type, exc_val, exc_tb):
    self.close()
  
  def _is_indexed_format(self, name):
    """
    检查字符串是否符合 "名称-页码" 格式
    
    Args:
      name: 待检查的字符串
    
    Returns:
      bool: 如果符合格式返回 True，否则返回 False
    """
    if not isinstance(name, str) or not name:
      return False
    # 检查是否以 "-数字" 结尾
    return bool(re.search(r'-\d+$', name))
  
  def _validate_entry_ref(self, name):
    """
    验证条目引用是否存在于索引中
    仅当 name 符合 "名称-页码" 格式时才检查
    
    Args:
      name: 条目名称字符串
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    # 如果不是索引格式，跳过检查
    if not self._is_indexed_format(name):
      return None
    
    # 检查是否在索引中
    if name not in self.indexes:
      return {"error": f"引用的条目不存在于索引中: {name}"}
    return None
  
  def _validate_string_list(self, items, field_name):
    """
    验证字符串列表中的所有条目引用
    
    Args:
      items: 字符串列表
      field_name: 字段名（用于错误提示）
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    if not isinstance(items, list):
      return {"error": f"字段 {field_name} 必须为 list 类型"}
    
    for i, item in enumerate(items):
      if not isinstance(item, str):
        return {"error": f"字段 {field_name} 的元素 [{i}] 必须为 str 类型"}
      
      # 只检查符合索引格式的字符串
      err = self._validate_entry_ref(item)
      if err is not None:
        return {"error": f"字段 {field_name} 的元素 [{i}] 引用无效: {err['error']}"}
    
    return None
  
  def _validate_staffing_list(self, items):
    """
    验证 staffing 列表中的所有条目引用
    staffing 格式：[[职位名称(str), 类别(str), 编制数量(num)], ...]
    
    Args:
      items: staffing 列表
    
    Returns:
      dict: 如果验证失败返回 {"error": "..."}, 否则返回 None
    """
    if not isinstance(items, list):
      return {"error": "字段 staffing 必须为 list 类型"}
    
    for i, item in enumerate(items):
      if not isinstance(item, list):
        return {"error": f"字段 staffing 的元素 [{i}] 必须为 list 类型（三元组）"}
      if len(item) != 3:
        return {"error": f"字段 staffing 的元素 [{i}] 必须包含 3 个元素（职位名称, 类别, 编制数量）"}
      
      position_name = item[0]
      if not isinstance(position_name, str):
        return {"error": f"字段 staffing 的元素 [{i}] 的职位名称必须为 str 类型"}
      
      # 只检查符合索引格式的职位名称
      err = self._validate_entry_ref(position_name)
      if err is not None:
        return {"error": f"字段 staffing 的元素 [{i}] 职位名称引用无效: {err['error']}"}
    
    return None
  
  # ========== 读取工具 ==========
  
  def search_dictionary(self, title, page):
    """查询《辞典》数据"""
    return search_dictionary(self._dict_conn, title, page)
  
  def check_existing_entry(self, title, page):
    """查询已有官制条目数据"""
    return check_existing_entry(self._entry_conn, title, page)
  
  # ========== 写入工具 ==========
  
  def insert(self, title, page, bgn_time, end_time, attr_key, attr_value):
    """填入官制条目的指定属性（创建）"""
    # 对于引用字段，需要额外验证
    if attr_key == "department":
      if attr_value is not None and attr_value != "":
        # department 是字符串
        err = self._validate_entry_ref(attr_value)
        if err is not None:
          return err
    
    elif attr_key in ["superiors", "subordinates"]:
      # 字符串列表
      err = self._validate_string_list(attr_value, attr_key)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      # staffing 是三元组列表
      err = self._validate_staffing_list(attr_value)
      if err is not None:
        return err
    
    return insert(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value)
  
  def update(self, title, page, bgn_time, end_time, attr_key, attr_value):
    """更新官制条目的指定属性（覆盖）"""
    # 对于引用字段，需要额外验证
    if attr_key == "department":
      if attr_value is not None and attr_value != "":
        err = self._validate_entry_ref(attr_value)
        if err is not None:
          return err
    
    elif attr_key in ["superiors", "subordinates"]:
      err = self._validate_string_list(attr_value, attr_key)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      err = self._validate_staffing_list(attr_value)
      if err is not None:
        return err
    
    return update(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value)
  
  def append_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index=None):
    """插入官制条目的指定列表属性（添加）"""
    # 对于引用列表字段，需要验证新增的元素
    if attr_key in ["superiors", "subordinates"]:
      if not isinstance(attr_value, str):
        return {"error": f"字段 {attr_key} 的元素必须为 str 类型"}
      err = self._validate_entry_ref(attr_value)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      if not isinstance(attr_value, list) or len(attr_value) != 3:
        return {"error": "staffing 的元素必须为三元组 list [职位名称, 类别, 编制数量]"}
      if not isinstance(attr_value[0], str):
        return {"error": "staffing 元素的职位名称必须为 str 类型"}
      err = self._validate_entry_ref(attr_value[0])
      if err is not None:
        return err
    
    return append_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def update_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index):
    """修改官制条目的指定列表属性（修改）"""
    # 对于引用列表字段，需要验证更新后的元素
    if attr_key in ["superiors", "subordinates"]:
      if not isinstance(attr_value, str):
        return {"error": f"字段 {attr_key} 的元素必须为 str 类型"}
      err = self._validate_entry_ref(attr_value)
      if err is not None:
        return err
    
    elif attr_key == "staffing":
      if not isinstance(attr_value, list) or len(attr_value) != 3:
        return {"error": "staffing 的元素必须为三元组 list [职位名称, 类别, 编制数量]"}
      if not isinstance(attr_value[0], str):
        return {"error": "staffing 元素的职位名称必须为 str 类型"}
      err = self._validate_entry_ref(attr_value[0])
      if err is not None:
        return err
    
    return update_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def remove_list(self, title, page, bgn_time, end_time, attr_key, attr_value, index):
    """删除官制条目的指定列表属性（删除）"""
    # remove_list 不需要额外的引用验证，因为是删除操作
    return remove_list(self._entry_conn, title, page, bgn_time, end_time, attr_key, attr_value, index)
  
  def new_time_point(self, title, page, bgn_time, end_time, time_point, event):
    """通过添加时间点的方式，拆分官制条目"""
    return new_time_point(self._entry_conn, title, page, bgn_time, end_time, time_point, event)


In [ ]:
"""
测试
会影响数据库中的数据
"""

# 1. 初始化 Tools 实例
tools = Tools(dict_db_path, entry_db_path, entry_indexes)

# 2. 查询《辞典》数据
result = tools.search_dictionary("河北兵马大元帅府", 482)
print("\n查询《辞典》数据:")
print(json.dumps(result, ensure_ascii=False, indent=2))

# 3. 查询官制条目数据
result = tools.check_existing_entry("河北兵马大元帅府", 482)
print("\n查询官制条目数据:")
if isinstance(result, list):
  print(f"找到 {len(result)} 条记录")
  if result:
    print(json.dumps(result[0], ensure_ascii=False, indent=2))
else:
  print(json.dumps(result, ensure_ascii=False, indent=2))

# 4. 测试写入操作（insert）
result = tools.insert("河北兵马大元帅府", 482, "-INF", "INF", "type", "机构")
print("\n插入 type 字段:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - type={result['type']}")

# 5. 测试 department 引用验证（有效引用）
result = tools.insert("河北兵马大元帅", 482, "-INF", "INF", "department", "河北兵马大元帅府-482")
print("\n插入 department 字段（有效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 6. 测试 department 引用验证（无效引用）
result = tools.insert("河北兵马元帅", 482, "-INF", "INF", "department", "不存在的机构-999")
print("\n插入 department 字段（无效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 7. 测试 department 引用验证（非索引格式，跳过检查）
result = tools.insert("河北兵马副元帅", 482, "-INF", "INF", "department", "大元帅府")
print("\n插入 department 字段（非索引格式，跳过检查）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - department={result['department']}")

# 8. 测试 superiors 字段（字符串列表）
result = tools.insert(
  "河北兵马大元帅府", 482, "-INF", "INF", 
  "subordinates", 
  [
    "河北兵马大元帅-482",
    "河北兵马元帅-482",
    "河北兵马副元帅-482"
  ]
)
print("\n插入 subordinates 字段（字符串列表，有效引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - subordinates 包含 {len(result['subordinates'])} 个条目")

# 9. 测试 staffing 字段（三元组列表）
result = tools.insert(
  "河北兵马大元帅府", 482, "-INF", "INF",
  "staffing",
  [
    ["河北兵马大元帅-482", "军职", 1],
    ["河北兵马元帅-482", "军职", 1],
    ["参议官", "幕职官", 2]  # 非索引格式，跳过检查
  ]
)
print("\n插入 staffing 字段（三元组列表，混合引用）:")
if "error" in result:
  print(f"错误: {result['error']}")
else:
  print(f"成功: {result['title']} - staffing 包含 {len(result['staffing'])} 个职位")

# 10. 关闭连接
tools.close()
print("\n已关闭所有数据库连接")


查询《辞典》数据:
{
  "id": 1,
  "title": "河北兵马大元帅府",
  "catalog": "宋代官制辞典/I.职官条目分类目录/第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门",
  "page": "482",
  "text": "官司名。北宋靖康元年闰十一月，宋钦宗传檄，授命康王为河北兵马大元帅。十二月一日，赵构开大元帅府，以募兵勤王抗金，解救京师之围为名（《要录》卷1）。南宋建炎元年五月十日大元帅府解散（《宋会要·职官》37之2《元帅府》）。",
  "fields": {
    "简称": "①大元帅府、元帅府。《宋会要·职官》37之1：“高宗建炎元年五月二日，诏大元帅府限十日结局。”《要录》卷1靖康元年十一月己酉：“拜王（康王赵构)河北兵马大元帅。”十二月壬戌朔：“王开元帅府。”②帅府。《要录》卷1乙亥：“乃遣人伴送至帅府。”③霸府。《要录》卷1，靖康元年闰十一月己酉：“拜王河北兵马大元帅。”原注引《汪伯彦日历》：“然霸府肇启开，事出仓卒。盖靖康元年闰十一月，檄到日，康王可充兵马大元帅。”④天下兵马大元帅府。过称。《金佗粹编》卷4《行实编年》：“（靖康元年）冬，高宗皇帝以天下兵马大元帅开府河朔。”《要录》卷4，建炎元年四月癸亥：“（赵子崧）望大王遵故事，以天下兵马大元帅承制号召四方。”"
  }
}

查询官制条目数据:
找到 1 条记录
{
  "id": 1,
  "title": "河北兵马大元帅府",
  "catalog": "宋代官制辞典/I.职官条目分类目录/第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门",
  "page": "482",
  "type": null,
  "subtype": null,
  "bgn_time": "-INF",
  "bgn_event": "始置",
  "end_time": "INF",
  "end_event": "罢置",
  "superiors": [],
  "subordinates": [],
  "staffing": [],
  "department": null,
  "grade": null
}

插入 type 字段:
成功: 河北兵马大元帅府 - type=机构

插入 department 字段（有